### github urlwith clone their files 

In [ ]:
### import all usefull libraries

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings 
from langchain_chroma import Chroma 

In [16]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
### for github handling

import os
from git import Repo


In [9]:
import os
from git import Repo
from langchain_community.document_loaders import TextLoader

# ==========================
# GitHub Repository URL
# ==========================

repo_url = input("Enter GitHub Repository URL: ").strip()

# Extract repo name
repo_name = repo_url.split("/")[-1]

# Local storage path
repo_path = f"./repos/{repo_name}"

# Create repos directory
os.makedirs("repos", exist_ok=True)

# ==========================
# Clone Repository
# ==========================

if not os.path.exists(repo_path):
    print("Cloning repository...")
    Repo.clone_from(repo_url, repo_path)
    print("✅ Repository cloned successfully")
else:
    print("✅ Repository already exists")

# ==========================
# Load Files
# ==========================

documents = []

py_count = 0
md_count = 0
txt_count = 0

supported_extensions = (".py", ".md", ".txt")

for root, dirs, files in os.walk(repo_path):

    for file in files:

        if file.endswith(supported_extensions):

            file_path = os.path.join(root, file)

            # Count file types
            if file.endswith(".py"):
                py_count += 1

            elif file.endswith(".md"):
                md_count += 1

            elif file.endswith(".txt"):
                txt_count += 1

            try:

                loader = TextLoader(
                    file_path,
                    encoding="utf-8"
                )

                docs = loader.load()

                # Add useful metadata
                for doc in docs:

                    doc.metadata["file_name"] = file
                    doc.metadata["repo_name"] = repo_name

                documents.extend(docs)

            except Exception as e:

                print(f"❌ Skipped: {file_path}")
                print(f"Reason: {e}")

# ==========================
# Summary
# ==========================

print("\n" + "=" * 50)
print("REPOSITORY SUMMARY")
print("=" * 50)

print(f"Repository Name : {repo_name}")
print(f"Python Files    : {py_count}")
print(f"Markdown Files  : {md_count}")
print(f"Text Files      : {txt_count}")
print(f"Documents Loaded: {len(documents)}")

# ==========================
# Preview
# ==========================

if documents:

    print("\nSample Metadata:")
    print(documents[0].metadata)

    print("\nSample Content:")
    print(documents[0].page_content[:500])

else:
    print("No documents loaded.")

✅ Repository already exists

REPOSITORY SUMMARY
Repository Name : ANN-Churn-Prediction
Python Files    : 1
Markdown Files  : 1
Text Files      : 1
Documents Loaded: 3

Sample Metadata:
{'source': './repos/ANN-Churn-Prediction\\app.py', 'file_name': 'app.py', 'repo_name': 'ANN-Churn-Prediction'}

Sample Content:
import streamlit as st
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pandas as pd
import pickle

# Load the trained model
model = tf.keras.models.load_model('model.h5')

# Load the encoders and scaler
with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scalar.pkl',


In [8]:
print(type(documents[0]))

<class 'langchain_core.documents.base.Document'>


### chunking

In [10]:
splitters=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [11]:
chunks=splitters.split_documents(documents)

In [12]:
chunks[0]

Document(metadata={'source': './repos/ANN-Churn-Prediction\\app.py', 'file_name': 'app.py', 'repo_name': 'ANN-Churn-Prediction'}, page_content='import streamlit as st\nimport numpy as np\nimport tensorflow as tf\nfrom sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder\nimport pandas as pd\nimport pickle\n\n# Load the trained model\nmodel = tf.keras.models.load_model(\'model.h5\')\n\n# Load the encoders and scaler\nwith open(\'label_encoder_gender.pkl\', \'rb\') as file:\n    label_encoder_gender = pickle.load(file)\n\nwith open(\'onehot_encoder_geo.pkl\', \'rb\') as file:\n    onehot_encoder_geo = pickle.load(file)\n\nwith open(\'scalar.pkl\', \'rb\') as file:\n    scalar = pickle.load(file)\n\n\nst.markdown("""\n<style>\n\n/* =========================\n   MAIN APP BACKGROUND\n========================= */\n.stApp {\n    background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);\n}\n\n/* =========================\n   GLOBAL TEXT (SAFE)\n=======================

In [14]:
print(len(documents))
print(len(chunks))

3
12


### embedding

In [18]:
embedding=OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [21]:
### test
test_embedding=embedding.embed_query("What is the purpose of this repository?")


### vector store 

In [24]:
vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./chroma_db",
)

In [26]:
### test
print(vector_store._collection.count())

12


### Retrievers 

In [54]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":8,
        "fetch_k":20
    }
)

In [30]:
## test 
docs = retriever.invoke(
    "What does this project do? "
)

print(docs[0].page_content)

# ANN-Churn-Prediction
A Ai powered Churn Prediction Ann Model project


### connect with llm 


In [53]:
model=ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.4
)

In [59]:
## prompt
prompt_template =PromptTemplate.from_template(
""" You are a senior software engineer and repository analyst.

Your task is to analyze a GitHub repository.

Instructions:
- Answer only from the provided repository context.
- If the user asks about tech stack, analyze dependencies, imports, and README.
- If the user asks about architecture, infer from the files.
- Mention file names whenever possible.
- Use bullet points.
- If information is missing, explicitly say so.

Repository Context:
{context}

Question:
{question}""")

In [60]:
def ask_question(question):

    # Retrieve docs
    retrieved_docs = retriever.invoke(question)

    # Context
    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # Build prompt
    prompt = prompt_template.format(
        context=context,
        question=question
    )

    # LLM call
    response = model.invoke(prompt)

    # Sources
    sources = list(
        set(
            doc.metadata["file_name"]
            for doc in retrieved_docs
        )
    )

    return {
        "answer": response.content,
        "sources": sources
    }

In [62]:
result = ask_question(
    "What tech stack is used in this repository and what this project does?"
)

print(result["answer"])

print("\nSources:")
print(result["sources"])


The tech stack used in this repository includes:

- **Frameworks and Libraries:**
  - `streamlit`: For building the web application interface.
  - `numpy`: For numerical operations.
  - `pandas`: For data manipulation and analysis.
  - `scikit-learn`: For machine learning preprocessing and model evaluation.
  - `tensorflow-cpu`: For building and using the artificial neural network model.
  - `matplotlib`: For plotting and visualizing data.
  - `scikeras`: For integrating Keras with scikit-learn.

### Project Description:
- The project is an AI-powered churn prediction model using a simple Artificial Neural Network (ANN).
- It estimates the probability of customer churn based on various customer profile inputs, such as geography, gender, age, credit score, estimated salary, account balance, etc.
- The application provides a user interface for inputting customer data and displays the churn prediction results.

Sources:
['app.py', 'requirements.txt', 'README.md']


### summary generators

In [63]:
def generate_repository_summary():

    summary_question = """
    Analyze this repository and provide:

    1. Project Purpose
    2. Tech Stack
    3. Important Files
    4. Project Workflow
    5. Dependencies
    6. Key Features
    """

    return ask_question(summary_question)

In [ ]:
repo_summary = generate_repository_summary()



### 1. Project Purpose
- The project is designed for **Customer Churn Prediction** using an Artificial Neural Network (ANN).
- It estimates the probability of customer churn based on various input features such as demographics and account details.

### 2. Tech Stack
- **Frameworks and Libraries**:
  - Streamlit (for web application)
  - TensorFlow (for building the ANN model)
  - Scikit-learn (for data preprocessing)
  - NumPy (for numerical operations)
  - Pandas (for data manipulation)
  - Matplotlib (for plotting, if needed)
  - Scikeras (for Keras integration with Scikit-learn)

### 3. Important Files
- **`model.h5`**: Contains the trained ANN model.
- **`label_encoder_gender.pkl`**: Stores the label encoder for gender.
- **`onehot_encoder_geo.pkl`**: Stores the one-hot encoder for geography.
- **`scalar.pkl`**: Contains the scaler used for feature normalization.

### 4. Project Workflow
- User inputs their profile information through the Streamlit interface.
- The input data is pr

### complete analyser

In [65]:
def analyze_repository():

    questions = {
        "purpose":
        "What is the main purpose of this repository?",

        "tech_stack":
        "List the complete technology stack used in this repository.",

        "architecture":
        "Explain the repository architecture in detail.",

        "workflow":
        "Explain the complete workflow of the project.",

        "important_files":
        "List important files and explain their purpose.",

        "dependencies":
        "List important dependencies and why they are used."
    }

    report = {}

    for section, question in questions.items():

        result = ask_question(question)

        report[section] = result["answer"]

    return report

In [67]:
analysis = analyze_repository()

print(analysis["purpose"])

- The main purpose of this repository is to provide an AI-powered churn prediction model using an Artificial Neural Network (ANN).
- It allows users to input customer profile details and account information to predict the likelihood of customer churn.


In [68]:
for section, content in analysis.items():

    print("\n" + "="*60)
    print(section.upper())
    print("="*60)

    print(content)


PURPOSE
- The main purpose of this repository is to provide an AI-powered churn prediction model using an Artificial Neural Network (ANN).
- It allows users to input customer profile details and account information to predict the likelihood of customer churn.

TECH_STACK
The complete technology stack used in this repository includes the following dependencies and libraries:

- **Streamlit**: For building the web application interface.
- **NumPy**: For numerical operations and handling arrays.
- **Pandas**: For data manipulation and analysis.
- **Scikit-learn**: For machine learning preprocessing tools like StandardScaler, LabelEncoder, and OneHotEncoder.
- **TensorFlow (tensorflow-cpu)**: For building and training the artificial neural network (ANN) model.
- **Matplotlib**: For data visualization (not explicitly used in the provided code but mentioned in the context).
- **Scikeras**: For integrating Keras with Scikit-learn (not explicitly shown in the provided code but included in the

### ui